# HITL: multiple interrupts

Here in a graph flow we will get multiple interrupts message if the graph is have parallel nodes.
In this case we will get the List of `__interrupt__`

```python
Interrupt(value={'message': 'give the number for cube'}, id='eb2b0c5200b2a33059fb38e1dd59bc45')
Interrupt(value={'message': 'give the number for square'}, id='281d6583714d8399c1402c3d284dae4c')
```

to resume these type of workflow we need to invoke graph again with `Command(resume={id1: value1, id2: value2, ...})`
here in the resume we need to send an object which contains value of each interrupts:
```python
resume = {
    "eb2b0c5200b2a33059fb38e1dd59bc45": 3,
    "281d6583714d8399c1402c3d284dae4c": 5
}
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict

In [ ]:
class State(TypedDict):
    square: int
    cube: int
    message: str

In [ ]:
def square_node(state: State):
    decision = interrupt({
        "message":"give the number for square"
    })

    if not decision:
        return {
            "message": "square fn not approved"
        }
    return {
        "square": decision**2
    }

def cube_node(state: State):
    decision = interrupt({
        "message":"give the number for cube"
    })

    if not decision:
        return {
            "message": "cube fn not approved"
        }
    return {
        "cube": decision**3
    }

def accumulator(state: State):
    return {}

In [ ]:
checkpointer = InMemorySaver()

graph = StateGraph(State)\
    .add_node("square_node", square_node)\
    .add_node("cube_node", cube_node)\
    .add_node("accumulator", accumulator)\
    .add_edge(START, "square_node")\
    .add_edge(START, "cube_node")\
    .add_edge("square_node", "accumulator")\
    .add_edge("cube_node", "accumulator")\
    .add_edge("accumulator", END)\
    .compile(checkpointer=checkpointer)

In [ ]:
# graph

In [ ]:
config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

In [ ]:
# first time execution 

resp = graph.invoke({}, config)

In [ ]:
# handling multiple interrupts
resume = {}

for itr in resp['__interrupt__']:
    print(itr)

    v = int(input(itr.value['message']))
    resume[itr.id] = v

resume

Interrupt(value={'message': 'give the number for cube'}, id='58df8689466640f6ed77ba6656281a54')
Interrupt(value={'message': 'give the number for square'}, id='19afc406e7857d4dad7588077532af02')


{'58df8689466640f6ed77ba6656281a54': 7, '19afc406e7857d4dad7588077532af02': 9}

In [ ]:
# final execution with human response

final_resp = graph.invoke(Command(resume=resume), config)

print(final_resp)

{'square': 49, 'cube': 125}
